LIBRARIES

In [1]:
import os
os.environ["WANDB__SERVICE"] = "0"

import wandb

wandb.login(key="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", relogin=True)

from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\pmkha\_netrc
wandb: Currently logged in as: masoodpk (masoodpk-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WANDB CONFIGURATION

In [2]:
sweep_config= {
    'method' :'grid',
    'metric' : {
        'name' : 'val_accuracy',
        'goal' : 'maximize'
    },
    'parameters' : {
        'epochs' : {'value': [5, 10]},
        'learning_rate' : {'value' : [0.01, 0.001, 0.0001]},
        'batch_size' : {'value' : [1, 16, 32]},
        'hidden_nodes' : {'value' : [64, 128, 256]},
        'img_size' : {'value' : [16, 64, 224]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="5flowers-new", entity="masoodpk-student")

Create sweep with ID: vbpfccbk
Sweep URL: https://wandb.ai/masoodpk-student/5flowers-new/sweeps/vbpfccbk


In [3]:
def train():
    with wandb.init(mode="offline") as run:
        config = wandb.config
        IMG_HEIGHT = config.img_size
        IMG_WIDTH = config.img_size
        IMG_CHANNELS = 3
        CLASS_NAMES = ["lilly", "lotus", "orchid", "sunflower", "tulip"]
        train_dataset = tf.keras.utils.image_dataset_from_directory(
            r"C:\My Folder\Projects\Computer_Vision\flower_images\train",
            labels = 'inferred',
            label_mode = 'int',
            batch_size = config.batch_size,
            image_size = (IMG_HEIGHT, IMG_WIDTH)
        )
        train_dataset = tf.keras.utils.image_dataset_from_directory(
            r"C:\My Folder\Projects\Computer_Vision\flower_images\val",
            labels = 'inferred',
            label_mode = 'int',
            batch_size = config.batch_size,
            image_size = (IMG_HEIGHT, IMG_WIDTH)
        )
        normalize = tf.keras.layers.Rescaling(1./255)
        train_dataset= train_dataset.map(lambda x, y: (normalize(x), y))
        val_dataset = val_dataset.map(lambda x, y: (normalize(x), y))

        model = keras.Sequential([
            keras.layers.Flatten(input_shape = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
            keras.layers.Dense(config.hidden_nodes, activation = 'relu'),
            keras.layers.Dense(len(CLASS_NAMES), activation = 'softmax')
        ])
        model.compile(
            optimizer = tf.keras.optimizers.Adam(learning_rate = config.learning_rate),
            losses = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = False),
            metrics = ['accuracy']
        )
        model.fit(
            train_dataset,
            validation_data = val_dataset,
            epochs = config.epochs,
            callbacks = [WandbMetricsLogger(log_freq=5)]
            )
    

In [4]:
wandb.agent(sweep_id, function =train)

wandb: Agent Starting Run: yaenyi0m with config:
wandb: 	batch_size: [1, 16, 32]
wandb: 	epochs: [5, 10]
wandb: 	hidden_nodes: [64, 128, 256]
wandb: 	img_size: [16, 64, 224]
wandb: 	learning_rate: [0.01, 0.001, 0.0001]
Traceback (most recent call last):
  File "C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\wandb\agents\pyagent.py", line 304, in _run_job
    self._function()
    ~~~~~~~~~~~~~~^^
  File "C:\Users\pmkha\AppData\Local\Temp\ipykernel_15316\3114589318.py", line 2, in train
    with wandb.init(mode="offline") as run:
         ~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\wandb\sdk\wandb_init.py", line 1595, in init
    get_sentry().reraise(e)
    ~~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\wandb\analytics\sentry.py", line 190, in reraise
    raise exc.with_traceback(tb)
  File "C:\Users\pmkha\AppData\Roaming\Python\Python313\site-packages\wandb\sdk\wandb_init.p